# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saadtalat111/flyrank/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

The Rule: If a piece of content is highly visible (ranking in the top 5 positions on Google) but has a suspiciously low Click-Through Rate (under 3.0%), it means users are seeing our link but choosing a competitor's. These pages represent quick-win opportunities where simply rewriting the title tag or meta description could instantly boost traffic.
Reason Code: HIGH_RANK_LOW_CTR
Action Label: UPDATE_META
Signal Verdicts:
Signal 1 (Avg Position vs CTR): CONFIRMED. As average position worsens, the median CTR drops, confirming position drives baseline CTR expectations.
Signal 2 (CTR Distribution on Page 1): CONFIRMED. Even on page 1, a significant volume of pages suffer from <3% CTR, confirming our target population exists.

In [17]:
import pandas as pd
import numpy as np

# 1. Load the starter dataset directly from the GitHub raw URL (Colab fix)
repo_url = 'https://raw.githubusercontent.com/saadtalat111/flyrank/main/data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(repo_url)

# GOTCHA FIX: avg_position = 0 means 'no data'. We must filter this out first!
df_clean = df[df['avg_position'] > 0].copy()

# --- Signal 1: Average Position vs. CTR Expectations ---
# We expect position 1-3 to have high CTR. If not, the signal is broken.
df_clean['position_bucket'] = pd.cut(df_clean['avg_position'], bins=[0, 3, 5, 10, 100], labels=['Top 3', 'Pos 4-5', 'Pos 6-10', 'Page 2+'])
pos_bucket = df_clean.groupby('position_bucket', observed=False).agg(
    n=('content_id', 'count'),
    median_ctr=('ctr', 'median')
)
print("--- Signal 1: Position vs Expected CTR ---")
print(pos_bucket)
print("\n")

# --- Signal 2: Identifying the Target Population ---
# Looking only at highly visible pages (Top 5) to see how many have terrible CTRs
top5_df = df_clean[df_clean['avg_position'] <= 5].copy()
top5_df['ctr_bucket'] = pd.cut(top5_df['ctr'], bins=[-1, 1.5, 3.0, 10.0, 100], labels=['Critical (<1.5%)', 'Low (1.5-3%)', 'Healthy (3-10%)', 'Excellent (>10%)'])
ctr_bucket = top5_df.groupby('ctr_bucket', observed=False).agg(
    n=('content_id', 'count'),
    avg_pos=('avg_position', 'mean')
)
print("--- Signal 2: CTR Distribution for Top-5 Pages ---")
print(ctr_bucket)

--- Signal 1: Position vs Expected CTR ---
                     n  median_ctr
position_bucket                   
Top 3             1141        0.00
Pos 4-5           2782        0.23
Pos 6-10          9060        0.14
Page 2+          15797        0.04


--- Signal 2: CTR Distribution for Top-5 Pages ---
                     n   avg_pos
ctr_bucket                      
Critical (<1.5%)  3584  3.623661
Low (1.5-3%)        89  3.898876
Healthy (3-10%)     99  3.785859
Excellent (>10%)   151  2.714570


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [18]:
import os

# Filter for our specific rule: Top 5 position AND less than 3.0% CTR
candidates = df_clean[(df_clean['avg_position'] <= 5) & (df_clean['ctr'] < 3.0)].copy()

# Define the Score: We want to prioritize Rank 1 pages with 0.5% CTR over Rank 5 pages with 2.9% CTR.
# Score formula: (6 - avg_position) * (3.0 - ctr)
# Example: Rank 1 with 1.0 CTR = (5) * (2.0) = 10.0 score
# Example: Rank 5 with 2.5 CTR = (1) * (0.5) = 0.5 score
candidates['score'] = (6 - candidates['avg_position']) * (3.0 - candidates['ctr'])

# Encode the rule labels
candidates['reason_code'] = 'HIGH_RANK_LOW_CTR'
candidates['action_label'] = 'UPDATE_META'

# Sort by highest score first
ranked_queue = candidates.sort_values('score', ascending=False)

# Clean up the output dataframe
final_output = ranked_queue[['content_id', 'client_id', 'avg_position', 'ctr', 'score', 'reason_code', 'action_label']]

# Write out the CSV to the requested path
os.makedirs('../outputs', exist_ok=True)
csv_path = '../outputs/baseline_action_score.csv'
final_output.to_csv(csv_path, index=False)

print(f"SUCCESS: Ranked queue built and saved to {csv_path}")
print(f"Total rows flagged: {len(final_output)}")
print("\nTop 5 rows:")
print(final_output.head(5))


SUCCESS: Ranked queue built and saved to ../outputs/baseline_action_score.csv
Total rows flagged: 3672

Top 5 rows:
                 content_id          client_id  avg_position  ctr  score  \
4873   content_38a55f070d18  client_7f2253d7e2           0.1  0.0   17.7   
18289  content_20d77f60fdb2  client_d4735e3a26           0.1  0.0   17.7   
26854  content_cfceaeb2ffa1  client_7f2253d7e2           0.1  0.0   17.7   
29156  content_7288a4d4c198  client_7f2253d7e2           0.1  0.0   17.7   
27056  content_5919b351bfb8  client_7f2253d7e2           0.2  0.0   17.4   

             reason_code action_label  
4873   HIGH_RANK_LOW_CTR  UPDATE_META  
18289  HIGH_RANK_LOW_CTR  UPDATE_META  
26854  HIGH_RANK_LOW_CTR  UPDATE_META  
29156  HIGH_RANK_LOW_CTR  UPDATE_META  
27056  HIGH_RANK_LOW_CTR  UPDATE_META  


## 3. Top-20 review
content_38a55f070d18: UPDATE_META. Why it's here: High rank (0.1) and 0.0% CTR generated a max score of 17.7. What would make it wrong: An average position of 0.1 is impossible for standard organic search (which starts at 1). This is likely a tracking glitch, a zero-click knowledge panel, or a sitelink. Rewriting the meta description will not yield clicks here.
content_20d77f60fdb2: UPDATE_META. Why it's here: High rank (0.1) and 0.0% CTR generated a max score of 17.7. What would make it wrong: Same as above. The sub-1.0 position indicates this is not a traditional search result that can be optimized via standard metadata updates.
content_cfceaeb2ffa1: UPDATE_META. Why it's here: High rank (0.1) and 0.0% CTR generated a max score of 17.7. What would make it wrong: It suffers from the same positional anomaly, likely generating impressions in a Google widget where users cannot actually click through.
content_7288a4d4c198: UPDATE_META. Why it's here: High rank (0.1) and 0.0% CTR generated a max score of 17.7. What would make it wrong: Same positional anomaly. The data reflects impressions without the possibility or intent of a click.
content_5919b351bfb8: UPDATE_META. Why it's here: High rank (0.2) and 0.0% CTR generated a score of 17.4. What would make it wrong: While slightly different, 0.2 is still a sub-1.0 position anomaly indicating non-standard search visibility.

In [19]:
# Displaying the top rows reviewed in the markdown above
print(final_output.head(10))

                 content_id          client_id  avg_position  ctr  score  \
4873   content_38a55f070d18  client_7f2253d7e2           0.1  0.0   17.7   
18289  content_20d77f60fdb2  client_d4735e3a26           0.1  0.0   17.7   
26854  content_cfceaeb2ffa1  client_7f2253d7e2           0.1  0.0   17.7   
29156  content_7288a4d4c198  client_7f2253d7e2           0.1  0.0   17.7   
27056  content_5919b351bfb8  client_7f2253d7e2           0.2  0.0   17.4   
29712  content_6849c1007a4f  client_d4735e3a26           0.3  0.0   17.1   
21680  content_97764b7c0914  client_d4735e3a26           0.3  0.0   17.1   
8060   content_0c329c3f2934  client_f369cb89fc           0.3  0.0   17.1   
3412   content_9c94ce9a26b5  client_f369cb89fc           0.3  0.0   17.1   
17618  content_175ea196d3e0  client_d4735e3a26           0.3  0.0   17.1   

             reason_code action_label  
4873   HIGH_RANK_LOW_CTR  UPDATE_META  
18289  HIGH_RANK_LOW_CTR  UPDATE_META  
26854  HIGH_RANK_LOW_CTR  UPDATE_META  
291

## 4. Weak picks + leakage check

Weak picks:
The absolute weakest picks in this baseline are at the very top of the ranked queue. My linear scoring formula (6 - avg_position) * (3.0 - ctr) heavily rewarded rows with an avg_position less than 1.0 (e.g., 0.1 or 0.2). Because standard organic search positions begin at 1, these sub-1.0 positions are tracking artifacts, zero-click widgets, or knowledge panels. They have an artificially high score, but standard SEO actions (like updating meta descriptions) will not fix their 0% CTR. In a refined model, avg_position < 1.0 should be filtered out entirely.
Leakage check:
No product flags or future windows leaked in. The rule relies strictly on trailing metrics (avg_position and ctr). I explicitly avoided trend_pct and trend_direction to ensure no target-label logic from is_declining_label contaminated the baseline.

In [20]:
# Proving the weak picks anomaly by showing how many sub-1.0 positions scored highly
weak_picks = final_output[final_output['avg_position'] < 1.0]
print(f"Total weak picks with impossible positions (< 1.0): {len(weak_picks)}")


Total weak picks with impossible positions (< 1.0): 76


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.